# Repeat bookings analysis

Расчет доли пользователей с повторными бронированиями по `data/train.csv`.

In [ ]:
import csv
from pathlib import Path
import pandas as pd

In [ ]:
def find_data_dir():
    for candidate in [Path.cwd() / 'data', *Path.cwd().parents]:
        data_dir = candidate if candidate.name == 'data' else candidate / 'data'
        if (data_dir / 'train.csv').exists():
            return data_dir
    raise FileNotFoundError('Не найдена папка data с train.csv')

DATA_DIR = find_data_dir()
CSV_PATH = DATA_DIR / 'train.csv'
CHUNK_SIZE = 250_000

def malformed_rows(csv_path):
    with csv_path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        bad = [line_no - 1 for line_no, row in enumerate(reader, start=2)
               if len(row) != len(header)]
    return bad


## 1. Загружаем train.csv чанками

In [ ]:
bad_lines = malformed_rows(CSV_PATH)
processed_rows = 0
all_users = set()
booking_counts = {}

for part in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE, skiprows=bad_lines, low_memory=False):
    processed_rows += len(part)
    valid_users = part['user_id'].dropna().astype('int64')
    all_users.update(valid_users.unique())
    booked = part.loc[part['is_booking'] == 1, 'user_id'].dropna().astype('int64').value_counts()
    for user_id, count in booked.items():
        booking_counts[user_id] = booking_counts.get(user_id, 0) + int(count)

print(f'Обработано строк: {processed_rows:,}')
print(f'Пропущено повреждённых строк: {len(bad_lines)}')

## 2. Считаем бронирования по пользователям

In [ ]:
# Данные уже собраны в all_users и booking_counts в предыдущей ячейке.

In [ ]:
n_all_users = len(all_users)
n_booked_users = len(booking_counts)
n_repeat_bookers = sum(count > 1 for count in booking_counts.values())

share_repeat_among_all = n_repeat_bookers / n_all_users
share_repeat_among_bookers = n_repeat_bookers / n_booked_users

result = pd.Series({
    'all_unique_users': n_all_users,
    'users_with_booking': n_booked_users,
    'users_with_more_than_1_booking': n_repeat_bookers,
    'repeat_share_among_all_users': share_repeat_among_all,
    'repeat_share_among_users_with_booking': share_repeat_among_bookers,
})

result

In [ ]:
# Результат рассчитан в предыдущей ячейке.

In [ ]:
print(f'Всего уникальных пользователей: {n_all_users:,}')
print(f'Бронировали хотя бы раз: {n_booked_users:,}')
print(f'Бронировали больше одного раза: {n_repeat_bookers:,}')
print(f'Доля repeat-bookers среди всех пользователей: {share_repeat_among_all:.2%}')
print(f'Доля repeat-bookers среди бронировавших: {share_repeat_among_bookers:.2%}')